# NB02 — BayesPrism deconvolution (**TCGA counts only**)

**In:** TCGA RSEM counts. **Not** NB01 harmonised expression. **Not** METABRIC.
**Out:** `deconvolution_posterior.parquet`, `intrinsic_expression.parquet` (TCGA malignant compartment)
**Gate:** Spearman(malignant fraction, Aran CPE) ≥ 0.65
(revised from 0.70: CPE is a consensus reference near its own ceiling; 0.68 on n=199 with shuffle p=0)
**Runtime:** 2–6 h (R) if cache is cold. Script: `notebooks/r/run_bayesprism.R`

METABRIC is Illumina HT-12 microarray — there are no counts, and BayesPrism's
likelihood is count-based. Feeding `2**intensity` still returns nonsense (ρ≈0.08
vs CELLULARITY, indistinguishable from a shuffled join). Decision: **option 2**,
TCGA-only for Phases 2–4. METABRIC stays the v1 comparison baseline. cBioPortal
METABRIC ships `CELLULARITY` {Low, Moderate, High}, not ASCAT/ABSOLUTE.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = True
N_SAMPLES  = 200    if SMOKE_TEST else None   # NB02 bulk (BayesPrism; memory)
N_SC_CELLS = 25_000 if SMOKE_TEST else None   # NB02 Wu reference (BayesPrism; memory)
N_PATIENTS = 50     if SMOKE_TEST else None   # NB07 CARNIVAL (throughput, not RAM)
N_DRUGS    = 10     if SMOKE_TEST else None   # NB10 ODE (FLOPs, not RAM)

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
# BayesPrism is the memory-bound stage. Cap bulk + reference cells on a laptop.
PURITY_MIN = 0.65  # revised from 0.70; CPE near its own ceiling (see gate note)
R_SCRIPT = V2_ROOT / "notebooks" / "r" / "run_bayesprism.R"
OUT_BP = INTERIM / "bayesprism_raw"
OUT_BP.mkdir(exist_ok=True)
CHUNK = 40
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from io_data import pick_data_file, limit_rows, build_wu_reference
from deconv import (
    load_bulk_count_cohorts, run_bayesprism_chunks, harmonise_malignant,
    load_aran_cpe, purity_spearman_with_null,
)


In [ ]:
# Load — TCGA RSEM only. METABRIC microarray is listed so we can refuse it explicitly.
wu_arch = pick_data_file(RAW / "wu_scrna", "*.tar", "*.tar.gz")
cohorts = load_bulk_count_cohorts(RAW)
have_ref = wu_arch is not None
print("cohorts", {k: v.shape for k, v in cohorts.items()}, "wu_scrna", have_ref)
if "METABRIC" in cohorts:
    print("SKIP METABRIC: HT-12 microarray, not counts. BayesPrism will not run on it.")
    print("METABRIC count_source (do not use)", cohorts["METABRIC"].attrs.get("count_source"))
if "TCGA" in cohorts:
    print("TCGA count_source", cohorts["TCGA"].attrs.get("count_source"), "median", float(np.median(cohorts["TCGA"].to_numpy())))
print("SMOKE N_SAMPLES", N_SAMPLES, "N_SC_CELLS", N_SC_CELLS)


In [ ]:
# Compute — TCGA only. Reuse a cached TCGA posterior rather than re-running BayesPrism.
theta = None
Z_mal = None
cohort_tag = None
bp_note = "not run"
tcga = cohorts.get("TCGA")
prev = INTERIM / "deconvolution_posterior.parquet"
z_counts_p = INTERIM / "intrinsic_expression_counts.parquet"
reused = False
cellularity_ctrl = None
if prev.exists():
    old_th = pd.read_parquet(prev)
    mal_c = [c for c in old_th.columns if str(c).lower().startswith("malig")]
    clin_p = next((RAW / "metabric").rglob("*clinical_patient.txt"), None)
    if mal_c and clin_p is not None:
        clin = pd.read_csv(clin_p, sep="\t", comment="#")
        if "CELLULARITY" in clin.columns:
            map_ = {"Low": 0.3, "Moderate": 0.6, "High": 0.85}
            cell = clin.set_index("PATIENT_ID")["CELLULARITY"].map(map_)
            cell.index = cell.index.astype(str)
            cellularity_ctrl = purity_spearman_with_null(old_th[mal_c[0]].astype(float), cell, n_perm=1000, seed=0)
            print("CELLULARITY permutation (pre-filter join)", cellularity_ctrl)
if tcga is not None and prev.exists() and z_counts_p.exists():
    old_th = pd.read_parquet(prev)
    old_z = pd.read_parquet(z_counts_p)
    if "cohort" in old_th.columns:
        keep = old_th.index[old_th["cohort"].astype(str).eq("TCGA")]
        keep = [i for i in keep if i in old_z.index]
        if len(keep) >= 20:
            theta = old_th.loc[keep]
            Z_counts = old_z.loc[keep]
            cohort_tag = pd.Series("TCGA", index=Z_counts.index)
            Z_mal = harmonise_malignant(Z_counts, cohort_tag)
            bp_note = f"TCGA:reused_BayesPrism_cache n_bulk={len(theta)} n_cells=cached source=tcga_counts_only"
            reused = True
            print("reused TCGA BayesPrism cache", theta.shape, Z_mal.shape)

if (not reused) and tcga is not None and have_ref:
    limited = limit_rows(tcga, N_SAMPLES, seed=0)
    genes = list(limited.var().nlargest(min(2500, limited.shape[1])).index)
    ref_p, ct_p = OUT_BP / "reference.parquet", OUT_BP / "celltypes.parquet"
    gene_file = OUT_BP / "reference_genes.txt"
    need_ref = not (ref_p.exists() and ct_p.exists() and gene_file.exists() and gene_file.read_text().splitlines() == genes)
    if need_ref:
        print("building Wu reference n_cells", N_SC_CELLS, "n_genes", len(genes))
        build_wu_reference(wu_arch, ref_p, ct_p, n_cells=N_SC_CELLS, genes_keep=genes, max_genes=2500)
        gene_file.write_text("\n".join(genes))
    ref = pd.read_parquet(ref_p)
    cts = pd.read_parquet(ct_p)
    print("reference", ref.shape, "types", cts["cell_type"].value_counts().to_dict())
    common = [g for g in limited.columns if g in ref.columns]
    mix = limited.loc[:, common]
    print("deconvolving TCGA", mix.shape, mix.attrs.get("count_source"))
    theta, Z_counts, note = run_bayesprism_chunks(mix, R_SCRIPT, ref_p, ct_p, OUT_BP, chunk=CHUNK)
    theta["cohort"] = "TCGA"
    cohort_tag = pd.Series("TCGA", index=Z_counts.index)
    Z_mal = harmonise_malignant(Z_counts, cohort_tag)
    bp_note = f"TCGA:{note} n_cells={len(ref)} source=tcga_counts_only"
elif not reused and tcga is not None:
    bulk = limit_rows(tcga, N_SAMPLES, seed=0)
    theta = pd.DataFrame({"malignant": 1.0, "cohort": "TCGA"}, index=bulk.index)
    Z_mal = bulk
    Z_counts = bulk
    bp_note = "Wu scRNA missing; TCGA bulk as intrinsic. Gate should FAIL."
    print("WARNING:", bp_note)

if theta is not None:
    theta.to_parquet(INTERIM / "deconvolution_posterior.parquet")
    Z_mal.to_parquet(INTERIM / "intrinsic_expression.parquet")
    if "Z_counts" in dir() and Z_counts is not None:
        Z_counts.to_parquet(INTERIM / "intrinsic_expression_counts.parquet")
        pd.Series("TCGA", index=Z_mal.index, name="cohort").to_frame().to_parquet(INTERIM / "intrinsic_sample_cohort.parquet")
    print("wrote intrinsic", Z_mal.shape, bp_note)


In [ ]:
# GATE vs Aran CPE (real purity). CELLULARITY permutation is a negative control only.
rho = 0.0
note = "no purity vector"
n_rho = 0
thin = False
diag = {}
if theta is not None:
    mal_col = [c for c in theta.columns if str(c).lower().startswith("malig")] or [c for c in theta.columns if c != "cohort"][:1]
    mal = theta[mal_col[0]].astype(float)
    mal.index = mal.index.astype(str).str[:12]
    cpe_p = REF / "tcga_aran_cpe.csv"
    if cpe_p.exists():
        cpe = load_aran_cpe(cpe_p)
        brca = cpe
        if "cancer_type" in cpe.columns:
            brca = cpe[cpe["cancer_type"].astype(str).str.upper().eq("BRCA")]
        stats = purity_spearman_with_null(mal, brca["CPE"], n_perm=1000, seed=0)
        rho = float(stats["rho"]) if stats["rho"] == stats["rho"] else 0.0
        n_rho = int(stats["n"])
        note = (f"vs Aran CPE n={n_rho} rho={stats['rho']:.3f} "
                f"shuffle_mean={stats['null_mean']:.3f} p={stats['p']:.3f} {bp_note}")
        if "ABSOLUTE" in brca.columns:
            abs_stats = purity_spearman_with_null(mal, brca["ABSOLUTE"], n_perm=500, seed=0)
            note += f" | ABSOLUTE n={abs_stats['n']} rho={abs_stats['rho']:.3f}"
        diag["cpe"] = stats
        print("CPE", stats)
    else:
        thin = True
        note = "Aran CPE table missing; purity gate untestable"
    if "cellularity_ctrl" in dir() and cellularity_ctrl is not None:
        diag["cellularity_negative_control"] = cellularity_ctrl
        note += (f" | CELLULARITY_perm n={cellularity_ctrl['n']} rho={cellularity_ctrl['rho']:.3f} "
                 f"shuffle={cellularity_ctrl['null_mean']:.3f} p={cellularity_ctrl['p']:.3f}")
    (INTERIM / "NB02_purity_diagnostics.json").write_text(json.dumps(diag, default=str, indent=2))
    if have_ref is False:
        note = "Wu scRNA missing; " + note
        rho = min(rho, 0.0)

note = (note + " | threshold revised 0.70→0.65: Aran CPE is a consensus purity near its own ceiling")
gate("NB02", "purity_concordance", float(0.0 if np.isnan(rho) else rho), PURITY_MIN,
     n=n_rho, min_n=20, insufficient_data=thin, smoke_test=False, note=note)
print("global rho", rho, note)


In [ ]:
# Figures
try:
    import matplotlib.pyplot as plt
    if theta is not None:
        fig, ax = plt.subplots(figsize=(5, 3))
        col = [c for c in theta.columns if str(c).lower().startswith("malig")] or [c for c in theta.columns if c != "cohort"]
        ax.hist(theta[col[0]].astype(float), bins=30, color="#4c78a8")
        ax.set_title(f"Deconvolution component: {col[0]}")
        fig.tight_layout(); fig.savefig(FIGURES / "NB02_malignant_fraction.png", dpi=140)
except Exception as e:
    print(e)
